# Introduzione ad Astropy

Astropy è una libreria Python progettata per l'astronomia. Fornisce strumenti per:
- **Coordinate celesti**: gestione di sistemi di coordinate (ICRS, FK5, Galactic, ecc.)
- **Unità fisiche**: conversione e calcolo con unità astronomiche
- **Costanti astronomiche**: velocità della luce, costante gravitazionale, massa solare, ecc.
- **File FITS**: lettura, scrittura e manipolazione di file FITS (Flexible Image Transport System)
- **Spettroscopia**: analisi di dati spettrali

In questo notebook esploreremo le funzionalità principali di Astropy.

## 1. Sistemi di coordinate celesti

`SkyCoord` permette di rappresentare coordinate celesti in vari sistemi di riferimento. Le coordinate equatoriali (RA, Dec) sono espresse nel sistema ICRS (International Celestial Reference System).

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u

def crea_coordinate(ra: float, dec: float, frame: str = 'icrs') -> SkyCoord:
    """Crea un oggetto SkyCoord da coordinate equatoriali.

    Args:
        ra: Ascensione retta in gradi
        dec: Declinazione in gradi
        frame: Sistema di coordinate (default: 'icrs')

    Returns:
        Oggetto SkyCoord con le coordinate specificate
    """
    return SkyCoord(ra=ra * u.degree, dec=dec * u.degree, frame=frame)

# Coordinate di M87 (galassia ellittica gigante)
coord = crea_coordinate(ra=187.705931, dec=12.391123)
print(f"Coordinate M87 (ICRS):")
print(f"  RA:  {coord.ra}")
print(f"  DEC: {coord.dec}")

## 2. Gestione delle unità fisiche

Astropy semplifica la gestione delle unità fisiche e delle conversioni tra sistemi di misura differenti.

In [ ]:
from astropy import units as u

def converti_massa(massa_kg: float, unita_destinazione: str = 'g') -> u.Quantity:
    """Converte una massa da kg all'unità specificata.

    Args:
        massa_kg: Massa in chilogrammi
        unita_destinazione: Unità di destinazione (es. 'g', 'M_sun', 'M_earth')

    Returns:
        Quantità convertita
    """
    massa = massa_kg * u.kg
    return massa.to(unita_destinazione)

m_terra = 5.972e24  # kg
print(f"Massa terrestre: {converti_massa(m_terra, 'g'):.2e}")
print(f"Massa terrestre in masse solari: {converti_massa(m_terra, 'M_sun'):.6e}")

## 3. Costanti astronomiche e calcoli orbitali

Il modulo `astropy.constants` contiene le costanti fisiche e astronomiche più importanti. Possiamo usarle per calcoli orbitali.

In [ ]:
from astropy.constants import G, M_sun, M_earth, c
from astropy import units as u
import numpy as np

def velocita_orb(raggio: float, unita: str = 'AU') -> u.Quantity:
    """Calcola la velocità orbitale circolare attorno al Sole.

    Args:
        raggio: Distanza dal Sole
        unita: Unità della distanza (default: 'AU')

    Returns:
        Velocità orbitale in km/s
    """
    r = raggio * u.Unit(unita)
    v = np.sqrt(G * M_sun / r)
    return v.to(u.km / u.s)

print(f"Velocità orbitale terrestre: {velocita_orb(1):.3f}")
print(f"Velocità orbitale di Mercurio (0.387 AU): {velocita_orb(0.387):.3f}")
print(f"Velocità orbitale di Nettuno (30.07 AU): {velocita_orb(30.07):.3f}")

## 4. Spettroscopia: dalla lunghezza d'onda alla frequenza

Astropy permette di lavorare con grandezze spettroscopiche. Usando la costante `c` (velocità della luce) possiamo convertire lunghezze d'onda in frequenze.

In [ ]:
from astropy.constants import c
from astropy import units as u

def lunghezza_a_frequenza(lunghezza_nm: float) -> u.Quantity:
    """Converte una lunghezza d'onda in frequenza.

    Args:
        lunghezza_nm: Lunghezza d'onda in nanometri

    Returns:
        Frequenza corrispondente in Hz
    """
    lunghezza = lunghezza_nm * u.nm
    frequenza = c / lunghezza
    return frequenza.to(u.Hz)

# Righe spettrali note
righe = {
    'H-alpha (656.3 nm)': 656.3,
    'H-beta (486.1 nm)': 486.1,
    'O III (500.7 nm)': 500.7,
    'Na D (589.0 nm)': 589.0
}

for nome, l in righe.items():
    f = lunghezza_a_frequenza(l)
    print(f"{nome} -> Frequenza: {f:.3e}")

## 5. Lavorare con file FITS

Il formato FITS (Flexible Image Transport System) è lo standard per la distribuzione di dati astronomici. Astropy fornisce il modulo `astropy.io.fits` per leggere, scrivere e manipolare questi file.

In [ ]:
import os
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

def analizza_fits(filepath: str):
    """Analizza un file FITS e ne mostra il contenuto.

    Args:
        filepath: Percorso del file FITS
    """
    if not os.path.exists(filepath):
        print(f"ERRORE: File '{filepath}' non trovato.")
        return

    try:
        with fits.open(filepath, memmap=False) as hdul:
            hdul.info()
            dati = hdul[0].data
            header = hdul[0].header

            print(f"\nDimensioni dati: {dati.shape}")
            print(f"Tipo dati: {dati.dtype}")
            print(f"\nStatistiche immagine:")
            print(f"  Min: {np.min(dati):.2f}")
            print(f"  Max: {np.max(dati):.2f}")
            print(f"  Media: {np.mean(dati):.2f}")
            print(f"  Dev. std: {np.std(dati):.2f}")

            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.imshow(dati, cmap='gray', origin='lower')
            plt.colorbar(label='Intensità')
            plt.title('Immagine FITS')

            plt.subplot(1, 2, 2)
            plt.hist(dati.flatten(), bins='auto', color='steelblue', edgecolor='black')
            plt.xlabel('Intensità pixel')
            plt.ylabel('Frequenza')
            plt.title('Istogramma intensità')
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"ERRORE durante l'apertura del file FITS: {e}")

# Prova ad aprire un file FITS se presente
percorsi = ['./data/HorseHead.fits', './data/M87_NEW.fits', './M87_NEW.fits']
for p in percorsi:
    if os.path.exists(p):
        print(f"Analisi del file: {p}")
        analizza_fits(p)
        break
else:
    print("Nessun file FITS trovato nei percorsi predefiniti.")

## Riepilogo

In questo notebook abbiamo esplorato:
- **SkyCoord**: coordinate celesti e sistemi di riferimento
- **astropy.units**: gestione e conversione di unità fisiche
- **astropy.constants**: costanti astronomiche per calcoli orbitali
- **astropy.io.fits**: lettura e analisi di file FITS
- **Spettroscopia**: conversione lunghezza d'onda-frequenza

Astropy è uno strumento fondamentale per l'analisi dati in ambito astronomico.